# Deep Past Initiative: Akkadian → English Machine Translation

This notebook implements a 3-stage neural machine translation pipeline for translating Old Assyrian Akkadian transliterations into English.

In [ ]:
# Project initialization and setup
import os
import sys
import pandas as pd
import numpy as np
import random
from sklearn.model_selection import train_test_split
import unicodedata
import re
import json

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)

# Create necessary directories
os.makedirs('data', exist_ok=True)
os.makedirs('models/stage1_final', exist_ok=True)
os.makedirs('models/stage2_final', exist_ok=True)
os.makedirs('models/stage3_final', exist_ok=True)
os.makedirs('outputs', exist_ok=True)

print("Project structure initialized successfully!")

## Data Loading and Initial Exploration

In [ ]:
# Load competition datasets
train_df = pd.read_csv('data/competition/train.csv')
test_df = pd.read_csv('data/competition/test.csv')
sample_submission_df = pd.read_csv('data/competition/sample_submission.csv')
sentences_oare_df = pd.read_csv('data/competition/Sentences_Oare_FirstWord_LinNum.csv')
published_texts_df = pd.read_csv('data/competition/published_texts.csv')
lexicon_df = pd.read_csv('data/competition/OA_Lexicon_eBL.csv')
dictionary_df = pd.read_csv('data/competition/eBL_Dictionary.csv')

print("Competition datasets loaded successfully!")
print(f"Train shape: {train_df.shape}")
print(f"Test shape: {test_df.shape}")
print(f"Sample submission shape: {sample_submission_df.shape}")
print(f"Sentences_Oare shape: {sentences_oare_df.shape}")
print(f"Published texts shape: {published_texts_df.shape}")
print(f"Lexicon shape: {lexicon_df.shape}")
print(f"Dictionary shape: {dictionary_df.shape}")

## Load External Datasets

In [ ]:
# Load Akkademia dataset
def load_akkademia_dataset():
    akkademia_data = {}
    splits = ['train', 'valid', 'test']
    
    for split in splits:
        with open(f'data/akkademia/{split}.ak', 'r', encoding='utf-8') as f:
            akkadian_lines = [line.strip() for line in f.readlines()]
        with open(f'data/akkademia/{split}.en', 'r', encoding='utf-8') as f:
            english_lines = [line.strip() for line in f.readlines()]
            
        akkademia_data[split] = pd.DataFrame({
            'transliteration': akkadian_lines,
            'translation': english_lines
        })
        
    print("Akkademia dataset loaded successfully!")
    for split in splits:
        print(f"{split} set size: {len(akkademia_data[split])}")
        
    return akkademia_data

# Load ORACC dataset
oracc_df = pd.read_csv('data/oracc_kaggle/train.csv')
print(f"ORACC dataset loaded successfully! Shape: {oracc_df.shape}")
print("ORACC columns:", oracc_df.columns.tolist())

# Load Akkademia data
akkademia_data = load_akkademia_dataset()